# Sprint 5 — SVM (LinearSVC baseline)

The contrast model to the tree (Section 7 #11). Loads the **featured split** (bucketed port, 67 features) so it is comparable to the best tree (0.9990 acc / 0.9984 recall).

Two things make the SVM different from the tree:
- **Scaling is required.** The SVM treats each flow as a vector and measures distances, so features must be on a comparable scale (a byte-rate of 0 to ~2e9 would drown out a 0/1 flag). `StandardScaler`, **fit on train only** (a fitted op, Section 7 #10).
- This is `LinearSVC` (a linear boundary), the tractable "SVM at 2.26M rows" (a kernel SVM would be ~O(n^2), infeasible here).

No `class_weight` yet: baseline first (Section 7 #12).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

proc = Path('../data/processed/featured')
X_train = pd.read_parquet(proc / 'X_train.parquet')
X_test  = pd.read_parquet(proc / 'X_test.parquet')
y_train = pd.read_parquet(proc / 'y_train.parquet')['label_binary']
y_test  = pd.read_parquet(proc / 'y_test.parquet')['label_binary']
print('featured split:', X_train.shape, '|', X_test.shape)
print('malicious frac  ->  train: %.4f   test: %.4f' % (y_train.mean(), y_test.mean()))

featured split: (2264502, 67) | (566126, 67)
malicious frac  ->  train: 0.1970   test: 0.1970


## 1. Scale the features (StandardScaler, train-only)

The SVM works on each flow as a vector and measures distances, so a feature ranging 0 to ~2e9 would dominate a 0/1 flag. `StandardScaler` centers each feature to mean 0, std 1. It is a **fitted** op: fit on `X_train`, apply the same transform to `X_test` (Section 7 #10).

In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)      # learns mean/std from TRAIN only
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)         # test uses the TRAIN mean/std, not its own

# self-check: train is now ~0 mean / ~1 std; test is transformed with train stats
print('train scaled: mean %.3f  std %.3f' % (X_train_s.mean(), X_train_s.std()))
print('test  scaled: mean %.3f  std %.3f  (need not be exactly 0/1 -- uses train stats)' % (X_test_s.mean(), X_test_s.std()))

train scaled: mean -0.000  std 1.000
test  scaled: mean -0.000  std 1.010  (need not be exactly 0/1 -- uses train stats)


## 2. Train LinearSVC (baseline, no class_weight)

A linear boundary with the widest margin between the classes. `dual=False` because rows (2.26M) far outnumber features (67). No `class_weight` yet, this is the imbalance baseline (Section 7 #12).

In [3]:
from sklearn.svm import LinearSVC
from time import perf_counter

t0 = perf_counter()
svm = LinearSVC(dual=False, random_state=42, max_iter=3000).fit(X_train_s, y_train)
print('fit in %.1fs | iterations: %s' % (perf_counter() - t0, getattr(svm, 'n_iter_', 'n/a')))

fit in 235.0s | iterations: 29


## 3. Evaluate, and compare to the tree

Same lens as the tree: accuracy (test + train), the confusion matrix (FN = missed intrusion, FP = false alarm), and per-class recall/precision. Benchmark: the bucketed Decision Tree scored **0.9990 acc / 0.9984 recall**.

In [4]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, classification_report

y_pred = svm.predict(X_test_s)

print('accuracy  test : %.4f   (bucketed tree: 0.9990)' % accuracy_score(y_test, y_pred))
print('accuracy  train: %.4f' % accuracy_score(y_train, svm.predict(X_train_s)))

cm = confusion_matrix(y_test, y_pred)
print('\nconfusion matrix (rows = actual, cols = predicted):')
print(pd.DataFrame(cm, index=['actual benign', 'actual malicious'], columns=['pred benign', 'pred malicious']))
print('\nFN (missed intrusions): %d   |   FP (false alarms): %d' % (cm[1, 0], cm[0, 1]))
print('\n' + classification_report(y_test, y_pred, target_names=['benign', 'malicious'], digits=4))
print('recall (malicious): %.4f   (bucketed tree: 0.9984)' % recall_score(y_test, y_pred, zero_division=0))

accuracy  test : 0.9412   (bucketed tree: 0.9990)
accuracy  train: 0.9412

confusion matrix (rows = actual, cols = predicted):
                  pred benign  pred malicious
actual benign          433864           20733
actual malicious        12550           98979

FN (missed intrusions): 12550   |   FP (false alarms): 20733

              precision    recall  f1-score   support

      benign     0.9719    0.9544    0.9631    454597
   malicious     0.8268    0.8875    0.8561    111529

    accuracy                         0.9412    566126
   macro avg     0.8993    0.9209    0.9096    566126
weighted avg     0.9433    0.9412    0.9420    566126

recall (malicious): 0.8875   (bucketed tree: 0.9984)
